# 13.2 - LangGraph State & Nodes

**Phase:** 13 - LangGraph / Stateful Workflows

**Status:** VERIFIED

---

## 1. What Are We Solving?

The manual loop works but becomes verbose and error-prone as graphs grow. LangGraph's `StateGraph` provides a typed state container, node registration, declared edges, checkpoints, and visualization.

## 2. Why Does This Matter?

## 3. Prerequisites

Unit 13.1 (manual state machine), Python `TypedDict`.

## 4. Learning Objectives

By the end of this unit, you should be able to:
- Define a typed state schema with `TypedDict`
- Register nodes with `add_node` and wire edges with `add_edge`
- Compile a graph and invoke it with a starting state
- Visualize the graph structure

## 5. Mental Model

LangGraph's `StateGraph` is a blueprint for a vending machine: you declare all slots (state fields), install mechanisms (nodes), and wire routes (edges). The framework handles the loop, persistence and serialization.


## 6. Setup + Version Check

In [1]:
import os
from dotenv import load_dotenv
load_dotenv()
from langchain_groq import ChatGroq
from typing import TypedDict
from langgraph.graph import StateGraph, START, END

GROQ_MODEL = os.environ.get("GROQ_MODEL", "openai/gpt-oss-20b")


def llm(prompt: str) -> str:
    if not os.environ.get("GROQ_API_KEY"):
        return "mock: positive sentiment"
    try:
        return ChatGroq(model=GROQ_MODEL, temperature=0.0).invoke(prompt).content.strip()
    except Exception as e:
        return f"[llm-error: {type(e).__name__}]"


outs = []
def trace(msg):
    outs.append(msg); print(msg)


import importlib.metadata
trace("langgraph " + importlib.metadata.version("langgraph"))
trace("Groq model: " + GROQ_MODEL)
trace("GROQ_API_KEY present: " + str(bool(os.environ.get("GROQ_API_KEY"))))


langgraph 1.2.11
Groq model: openai/gpt-oss-20b
GROQ_API_KEY present: True


## 7. First Graph: Clean -> Classify

Two nodes, one edge each, typed state. Note the modern API: entry/exit are wired with `START`/`END` instead of `set_entry_point`/`set_finish_point`.

In [2]:
class TextState(TypedDict):
    raw_text: str
    cleaned: str
    sentiment: str | None


def clean(state):
    state["cleaned"] = state["raw_text"].strip().lower()
    return state


def classify(state):
    prompt = ("Classify sentiment as positive, negative, or neutral. Reply with ONE word.\n"
              f"Text: {state['cleaned']}")
    tag = llm(prompt).strip().lower()
    tag = next((t for t in ("positive", "negative", "neutral") if t in tag), "neutral")
    state["sentiment"] = tag
    return state


g = StateGraph(TextState)
g.add_node("clean", clean)
g.add_node("classify", classify)
g.add_edge(START, "clean")
g.add_edge("clean", "classify")
g.add_edge("classify", END)
app = g.compile()

result = app.invoke({"raw_text": "  The product works surprisingly well, I love it!  ",
                     "cleaned": "", "sentiment": None})
print(result)


{'raw_text': '  The product works surprisingly well, I love it!  ', 'cleaned': 'the product works surprisingly well, i love it!', 'sentiment': 'positive'}


## 8. Document QA as a Linear RAG Pipeline

A real pattern: detect topic -> retrieve context -> answer with an LLM. Each node does exactly one thing.

In [3]:
docs = {
    "langgraph": "LangGraph is a stateful orchestration library that models agent workflows as graphs of nodes and edges.",
    "rag": "RAG combines retrieval over a knowledge base with a generative model to ground answers in evidence.",
    "memory": "Checkpointers persist graph state so agents remember context across invocations.",
}


class QASlate(TypedDict):
    question: str
    topic: str
    context: str
    answer: str


def detect_topic(state):
    q = state["question"].lower()
    state["topic"] = ("rag" if "rag" in q else "memory" if "memory" in q else "langgraph")
    return state


def retrieve(state):
    state["context"] = docs[state["topic"]]
    return state


def answer(state):
    state["answer"] = llm(f"Context: {state['context']}\n\nQuestion: {state['question']}\n"
                                 "Answer briefly using the context.")
    return state


qa = StateGraph(QASlate)
for n in ("topic", "retrieve", "answer"):
    qa.add_node(n, {"topic": detect_topic, "retrieve": retrieve, "answer": answer}[n])
qa.add_edge(START, "topic")
qa.add_edge("topic", "retrieve")
qa.add_edge("retrieve", "answer")
qa.add_edge("answer", END)
qa_app = qa.compile()

out = qa_app.invoke({"question": "What is RAG and why use it?", "topic": "", "context": "", "answer": ""})
print("TOPIC :", out["topic"])
print("CTX   :", out["context"])
print("ANSWER:", out["answer"])


TOPIC : rag
CTX   : RAG combines retrieval over a knowledge base with a generative model to ground answers in evidence.
ANSWER: **RAG** (Retrieval‑Augmented Generation) is a technique that fuses a retrieval system with a generative language model.  
- **What it does**: It first pulls relevant passages from a knowledge base and then feeds those passages into the model to generate an answer that is grounded in that evidence.  
- **Why use it**: It boosts factual accuracy, reduces hallucinations, and allows the system to cite up‑to‑date, domain‑specific information.


## 9. Visualize the Graph

`draw_mermaid()` renders the structure as text you can paste into any Mermaid renderer.

In [4]:
print(qa_app.get_graph().draw_mermaid())

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	topic(topic)
	retrieve(retrieve)
	answer(answer)
	__end__([<p>__end__</p>]):::last
	__start__ --> topic;
	retrieve --> answer;
	topic --> retrieve;
	answer --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc




## Common Mistakes

- **Return `None` from a node** — the graph silently drops the update. Always `return state`.
- **Mutating state in the router** — routers must be side-effect free.
- **Forgetting the terminal condition** — cycles run forever without an iteration guard.
- **Typo in a state key** — `TypedDict` catches it at compile time; plain dicts do not.

## Debugging

| Symptom | Likely Cause | Fix |
|---|---|---|
| `KeyError` on state | Field name mismatch | Match state keys to the `TypedDict` exactly |
| Graph won't compile | Node/referenced name typo | Check every string passed to `add_node`/`add_edge` |
| Node output ignored | Node returns `None` or a partial dict | Always `return state` (or a merge-able partial) |
| Infinite loop | No convergence guard | Add `max_steps` to state and check it in the router |
| Wrong branch taken | Router priority bug | Unit-test the router on every input variant |

## Best Practices

- Define all state fields upfront with defaults in a `TypedDict`.
- Keep node functions pure and focused: one responsibility each.
- Name nodes descriptively (`retrieve`, `generate`, not `step1`).
- Always add an iteration guard on loops.
- Inspect the graph with `app.get_graph().draw_mermaid()`.

## Hands-On Practice

1. **Basic:** Rerun the examples with new inputs; verify the trace.
2. **Guided:** Add a node that validates output before terminating.
3. **Independent:** Build a 3-step pipeline (fetch -> process -> summarize) with a retry node.
4. **Realistic:** Turn the example into a multi-department support agent.
5. **Challenge:** Save/load the state dict to JSON and resume the workflow from a checkpoint.

## Exit Criteria

- You can explain and build the concept from scratch.
- You can debug the associated failure modes.
- You know when to reach for this tool vs. a plain function.
